# Catalog vs Real Sky Image Diagnostics

This notebook visualizes whether the real SkyView image matches `data/catalog.bin` under the assumed camera model.

Run the cells top-to-bottom. The heavy work is delegated to `scripts/run_catalog_real_diagnostics.py`, which uses only `numpy` and `Pillow`.

In [ ]:
from pathlib import Path
import csv
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "catalog_real_diagnostics"
SUMMARY = OUTPUT_DIR / "plate_sweep_summary.csv"
print(PROJECT_ROOT)

## Run Diagnostics

This compares the real centroid CSV against catalog projections for DEC sign and FOV hypotheses.
If it takes more than 10 seconds, wait for the script to finish; it writes PNG overlays and a summary CSV.

In [ ]:
import subprocess

subprocess.run([
    "python",
    str(PROJECT_ROOT / "scripts" / "run_catalog_real_diagnostics.py"),
], cwd=PROJECT_ROOT, check=True)
print("done")

## Plate-Fit Summary

High match count alone is not enough because a dense catalog can overfit 10 centroids. Also check `rank_corr`, `scale`, and `mean_error_px`.

In [ ]:
with SUMMARY.open(newline="") as file:
    rows = list(csv.DictReader(file))
rows

## Visual Overlays

Cyan circles are detected centroids. Red squares are transformed catalog stars. Green circles are matched centroids.

In [ ]:
for name in ["overlay_positive_10.png", "overlay_negative_10.png", "overlay_negative_20.png"]:
    path = OUTPUT_DIR / name
    print(path)
    display(Image(filename=str(path)))

## Current Interpretation

For the provided image, the strongest plate fit is usually DEC negative with FOV near 20 degrees, not FOV 10 degrees. That means the current real image does not validate the C identifier under a 10-degree pinhole camera model.

Before using this image as a real test case, get either:
- a raw image with known WCS / exact FOV / no overlays, or
- a generated catalog-backed image with known projection.